In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler,LabelEncoder,OneHotEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from scikeras.wrappers import KerasClassifier

In [2]:
dataset = pd.read_csv('Churn_Modelling.csv', header=0)
print(dataset.shape)
dataset.head()

(10000, 14)


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
#preprocessing the data
dataset.drop(['RowNumber','CustomerId','Surname'],axis=1,inplace=True)
dataset.head()

#only geography and gender are categorical features
print("geography: ",dataset['Geography'].unique())
print("gender: ",dataset['Gender'].unique())

#preprocessing the categorical features
dataset['Gender'] = LabelEncoder().fit_transform(dataset['Gender'])
geography_encoder = OneHotEncoder(sparse_output=False)
geography_new = pd.DataFrame(geography_encoder.fit_transform(dataset[['Geography']]),columns=geography_encoder.get_feature_names_out(['Geography']))

#concat all data
dataset = pd.concat([dataset.drop('Geography', axis=1),geography_new],axis=1)
dataset.head()

geography:  <ArrowStringArray>
['France', 'Spain', 'Germany']
Length: 3, dtype: str
gender:  <ArrowStringArray>
['Female', 'Male']
Length: 2, dtype: str


,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [4]:
# X and Y data
X = dataset.drop('Exited', axis=1)
Y = dataset['Exited']

# splitting the data into training and testing data
X_train,X_test,Y_train,Y_test = train_test_split(X,Y,test_size=0.2,random_state=42)

#feature scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("X_train shape: ",X_train.shape)
print("X_test shape: ",X_test.shape)
print("Y_train shape: ",Y_train.shape)
print("Y_test shape: ",Y_test.shape)

X_train shape:  (8000, 12)
X_test shape:  (2000, 12)
Y_train shape:  (8000,)
Y_test shape:  (2000,)


In [11]:
def create_model(neurons=16,layers=1):
    model = Sequential()
    model.add(Dense(neurons,activation='relu', input_shape=(X_train.shape[1],)))
    for l in range(layers-1):
        model.add(Dense(neurons,activation='relu'))
    model.add(Dense(1,activation='sigmoid'))
    model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
    return model

In [12]:
#Keras Classifer
model = KerasClassifier(neurons=32,layers=1,build_fn=create_model,epochs=50,batch_size=10,verbose=1)


In [13]:
#params
param = {
    'neurons' : [16,32,48],
    'layers' : [1,2],
    'epochs' : [50,100]
}

#gridsearch
gc = GridSearchCV(estimator=model,param_grid=param,n_jobs=-1,cv=3, error_score='raise')
res = gc.fit(X_train,Y_train)
print(res.best_score_, "*********", res.best_params_)

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_d

Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50


KeyboardInterrupt: 